# The Cold-Start Problem

Wiki reference for [the cold-start problem](https://ml-viz-ruby.vercel.app/wiki/cold-start-problem).

**The idea in one sentence.** A recommender has no interaction history for a brand-new user or
item, so you bridge the gap with **content features** (project them into the embedding space)
and **exploration** (a UCB-style bonus that deliberately surfaces under-shown items) — because
pure exploitation would leave cold items cold forever.

We build a content→embedding projection and a UCB scorer from scratch, **validate that content
recovers useful embeddings and that UCB favours cold items**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
rng = np.random.default_rng(8)

## 1 — Content-based projection for new items

In [ ]:
# Simulated embedding space (d=16) and content feature space (d_c=32)
d, d_c, n_items = 16, 32, 200
item_emb = rng.normal(size=(n_items, d))  # interaction-based embeddings (warm items)
item_emb /= np.linalg.norm(item_emb, axis=1, keepdims=True)
content_feats = rng.normal(size=(n_items, d_c))  # content features for all items

# Learn projection: W maps content features -> embedding space
# Using pseudo-inverse (in production: trained jointly with the rec model)
W_proj = np.linalg.lstsq(content_feats, item_emb, rcond=None)[0]  # (d_c, d)

def cold_start_embed(content):
    emb = content @ W_proj
    return emb / np.linalg.norm(emb)

# New item: content features only, no interaction history
new_item_content = rng.normal(size=d_c)
new_item_emb = cold_start_embed(new_item_content)

# Find similar warm items
sims = item_emb @ new_item_emb
top5 = np.argsort(-sims)[:5]
print("New item's most similar existing items:", top5.tolist())
print("Similarity scores:", sims[top5].round(4))

### Validate: content features recover a useful embedding

For a cold item with no interactions, we project its **content features** into the interaction-
embedding space via a learned map. The projected embedding should align with the item's true
interaction embedding far better than with a random item's. We confirm.

In [ ]:
cold = content_feats @ W_proj
cold_n = cold / np.linalg.norm(cold, axis=1, keepdims=True)
self_sim = np.mean([cold_n[i] @ item_emb[i] for i in range(n_items)])
rand_sim = np.mean([cold_n[i] @ item_emb[(i + 1) % n_items] for i in range(n_items)])
print(f'content->embedding cosine: self={self_sim:.3f}  vs random item={rand_sim:.3f}')
assert self_sim > rand_sim + 0.1, 'content-projected embeddings align with the true embedding, not a random one'
print('\n✅ content features bridge the cold-start gap until real interactions arrive')

## 2 — UCB1 exploration bonus for new items

In [ ]:
class ItemUCB:
    def __init__(self, n_items):
        self.n = np.zeros(n_items)           # times shown
        self.mu = np.zeros(n_items)          # estimated CTR
        self.t = 0                           # total rounds

    def score(self, item_id, relevance_score, c=1.0):
        """UCB score = relevance + exploration bonus."""
        ucb_bonus = c * np.sqrt(np.log(self.t + 1) / (self.n[item_id] + 1))
        return relevance_score + ucb_bonus

    def update(self, item_id, clicked):
        self.n[item_id] += 1
        self.mu[item_id] = (self.mu[item_id] * (self.n[item_id]-1) + clicked) / self.n[item_id]
        self.t += 1

n_catalog = 100
ucb = ItemUCB(n_catalog)
relevance = rng.uniform(0.1, 0.9, n_catalog)   # model's base relevance scores

# Simulate 500 recommendation rounds
for t in range(500):
    # Score all items
    scores = np.array([ucb.score(i, relevance[i]) for i in range(n_catalog)])
    chosen = np.argmax(scores)
    # Simulate click (true CTR = relevance)
    clicked = float(rng.random() < relevance[chosen])
    ucb.update(chosen, clicked)

print(f"Most shown items: {np.argsort(-ucb.n)[:5].tolist()}")
print(f"Items never shown: {(ucb.n == 0).sum()}")
print(f"Estimated CTR for top item: {ucb.mu[np.argmax(ucb.mu)]:.3f}")
print(f"True CTR for top item:      {relevance[np.argmax(ucb.mu)]:.3f}")

### Validate: UCB gives cold items an exploration bonus

The UCB score adds an exploration bonus $c\sqrt{\ln(t+1)/(n_i+1)}$ that is **larger for
under-shown items**. So at equal relevance, a never-shown (cold) item scores higher than a
frequently-shown one — the mechanism that gets cold items surfaced. We confirm.

In [ ]:
u = ItemUCB(n_items); u.t = 100
u.n[0] = 0     # cold: never shown
u.n[1] = 80    # warm: shown often
score_cold = u.score(0, relevance_score=0.0)
score_warm = u.score(1, relevance_score=0.0)
print(f'exploration bonus: cold item={score_cold:.3f}  warm item={score_warm:.3f}')
assert score_cold > score_warm, 'UCB gives under-shown (cold) items a larger exploration bonus'
print('\n✅ exploration deliberately surfaces cold items so they can accumulate data')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **no exploration** | greedy starves cold items in a feedback loop (demo) |
| **content ≠ behaviour** | content embeddings are a proxy, lower quality than interactions |
| **popularity bias** | exploitation entrenches already-popular items |
| **new-user cold start** | no history at all; use onboarding surveys / demographics |
| **exploration cost** | too much exploration hurts short-term metrics — tune $c$ |

Demo: greedy exploitation never surfaces cold items.

In [ ]:
# Why exploration is NOT optional: pure exploitation creates a feedback loop that starves cold
# items. A greedy policy always picks the current best-estimated item, so items that start with a
# zero estimate (cold, unshown) are NEVER chosen — they can never earn the data that would raise
# their estimate. We simulate greedy selection and confirm the cold items stay at zero exposure.
est_ctr = np.array([0.30, 0.0, 0.0])   # item 0 warm (known CTR); items 1,2 cold (unknown -> 0)
shown = np.zeros(3)
for _ in range(500):
    shown[int(np.argmax(est_ctr))] += 1   # greedy: always the current best
print(f'greedy exposure over 500 rounds: {shown}')
assert shown[1] == 0 and shown[2] == 0, 'pure exploitation never surfaces cold items -> they stay cold forever'
print('\nWithout exploration the cold-start problem is self-perpetuating -> UCB / epsilon-greedy break the loop.')

## ✏️ Your turn

In [ ]:
def onboarding_item_selection(item_emb, k=10):
    """
    Select k items for an onboarding preference survey that maximally
    cover the embedding space diversity.
    
    Strategy: greedy farthest-point sampling — iteratively pick the item
    that is farthest from all already-selected items (maximizing min distance).
    
    item_emb: (N, d) normalized item embeddings
    Returns: list of k item indices
    """
    # TODO(you): start with a random item, then greedily add the item
    # most distant from all already-selected items
    return ...

selected = onboarding_item_selection(item_emb, k=10)
print(f"Selected {len(selected)} items for onboarding: {selected}")
# Verify diversity: mean pairwise distance should be high
pairs = [(i,j) for i in selected for j in selected if i<j]
mean_dist = np.mean([1 - item_emb[i] @ item_emb[j] for i,j in pairs])
print(f"Mean pairwise distance: {mean_dist:.4f} (random baseline ≈ {1 - item_emb[:10] @ item_emb[:10].T * 0 + 0.5:.2f})")

<details><summary>Solution</summary>

```python
def onboarding_item_selection(item_emb, k=10):
    N = len(item_emb)
    selected = [rng.integers(N).item()]
    for _ in range(k - 1):
        # Distance from each candidate to the nearest selected item
        min_dists = np.array([
            1 - max(item_emb[i] @ item_emb[s] for s in selected)
            for i in range(N)
        ])
        min_dists[selected] = -1   # don't re-select
        selected.append(int(np.argmax(min_dists)))
    return selected
```
</details>

## Key takeaways

- **Content features bridge cold start:** project them into the embedding space to recommend
  items with no history (verified).
- **Exploration surfaces cold items:** a UCB bonus favours under-shown items (verified).
- **Pure exploitation starves cold items:** greedy selection never surfaces them (demo) —
  exploration is mandatory.
- **Content is a proxy:** cold-start embeddings are refined once real interactions arrive.